# SignalScope Analysis Report

**Dataset:** synthetic_random_v1  
**Source:** random  
**Type:** synthetic  


## Data Description

This dataset was generated using a synthetic data generating process (DGP). The signal is embedded directly into returns using a linear factor model.

**Model:** Linear Factor Model  
**Equation:** `r_{i,t} = β · s_{i,t} + ε_{i,t}`  

**Implications:**
- Predictive power is structurally embedded
- IC and Sharpe may be inflated
- Results are not representative of real market data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline


In [ ]:
# Dataset (first rows from the processed pipeline)
_rows = [
    {
        "date": "2020-01-01",
        "asset": "ASSET_000",
        "signal": 0.30471707975443135,
        "return": -0.010399841062404956
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_001",
        "signal": 0.7504511958064572,
        "return": 0.009405647163912139
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_002",
        "signal": -1.9510351886538364,
        "return": -0.013021795068623181
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_003",
        "signal": 0.12784040316728537,
        "return": -0.003162425923435822
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_004",
        "signal": -0.016801157504288795,
        "return": -0.008530439275735801
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_005",
        "signal": 0.8793979748628286,
        "return": 0.0077779193542894835
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_006",
        "signal": 0.06603069756121605,
        "return": 0.011272412069680328
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_007",
        "signal": 0.4675093422520456,
        "return": -0.008592924628832382
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_008",
        "signal": 0.36875078408249884,
        "return": -0.009588826008289988
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_009",
        "signal": 0.8784503013072725,
        "return": -0.000499259109862529
    },
    {
        "date": "2020-01-02",
        "asset": "ASSET_000",
        "signal": -0.18486236354526056,
        "return": -0.006809295444039414
    },
    {
        "date": "2020-01-02",
        "asset": "ASSET_001",
        "signal": 1.2225413386740303,
        "return": -0.0015452948206880217
    },
    {
        "date": "2020-01-02",
        "asset": "ASSET_002",
        "signal": -0.4283278221631072,
        "return": -0.003521335504882296
    },
    {
        "date": "2020-01-02",
        "asset": "ASSET_003",
        "signal": 0.5323091855533487,
        "return": 0.0036544406436407836
    },
    {
        "date": "2020-01-02",
        "asset": "ASSET_004",
        "signal": 0.4127326115959884,
        "return": 0.004308210030078827
    }
]

df = pd.DataFrame(_rows)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])

df.head()


In [ ]:
# Quantile Analysis — mean return per signal bucket
_labels = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
_values = [-0.001491, 0.000189, -0.00075, 0.000192, -8.3e-05]

if _labels and _values:
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ['#d73027' if v < 0 else '#1a9850' for v in _values]
    ax.bar(_labels, _values, color=colors, edgecolor='white')
    ax.set_title('Quantile Returns (mean forward return per bucket)')
    ax.set_xlabel('Signal Quantile')
    ax.set_ylabel('Mean Return')
    plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
    # Spread (Q5 - Q1): 0.001408
    plt.tight_layout()
    plt.show()
else:
    print('No quantile data available.')


In [ ]:
# Signal vs Return scatter plot with OLS trend line
if 'signal' in df.columns and 'return' in df.columns:
    plt.figure(figsize=(5, 4))
    plt.scatter(df['signal'], df['return'], alpha=0.6, color='steelblue', s=18)

    # Regression trend line
    try:
        z = np.polyfit(df['signal'], df['return'], 1)
        p = np.poly1d(z)
        _xs = np.linspace(df['signal'].min(), df['signal'].max(), 200)
        plt.plot(_xs, p(_xs), color='tomato', linewidth=1.5, label='OLS fit')
        plt.legend()
    except Exception:
        pass

    plt.xlabel('Signal')
    plt.ylabel('Return')
    plt.title('Signal vs Return')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print('signal or return column not found in df.')


*The **linear pattern** reflects strong factor exposure (beta-driven returns). The OLS trend line aligns with a high-beta, low-alpha regime.*


In [ ]:
# Factor Decomposition — OLS regression results
# Equation: return = alpha + beta * signal + epsilon

beta         = 5.561677029587142
alpha        = -0.06923479679323355
residual_std = 0.9897192690876024

print(f'Beta:         {beta:.4f}' if beta is not None else 'Beta:         N/A')
print(f'Alpha:        {alpha:.4f}' if alpha is not None else 'Alpha:        N/A')
print(f'Residual std: {residual_std:.4f}' if residual_std is not None else 'Residual std: N/A')

# Interpretation:
# High beta + near-zero alpha → signal is primarily factor-driven
# Positive alpha → signal carries independent predictive content


## Interpretation

The Information Coefficient (IC) of 0.0568 and Rank IC of 0.0492 are below the 0.1 threshold, indicating limited overall predictive power. The Quantile Spread (Q5 − Q1) of 0.0014 suggests a small economic magnitude in return separation across signal buckets. The Cross-Sectional IC (Stability) mean of 0.0798 combined with a relatively high standard deviation of 0.3586 indicates low and unstable temporal predictive power. The Factor Beta of 5.5617 is high, while the Factor Alpha (Intercept) is negative at -0.0692, implying the signal's returns are largely explained by factor exposure with limited independent alpha. Overall, the signal primarily reflects factor-driven returns rather than generating meaningful independent alpha.


## Signal Classification

**Type:** factor-driven  
**Confidence:** low  

This signal primarily captures **systematic factor exposure** rather than independent alpha.


## Conclusion

The signal appears to derive its performance from **factor exposure** rather than independent alpha. Returns are largely explained by systematic risk.  

**Classification:** factor-driven  
**Confidence:** low  
